# Qwen2.5-32B + Open WebUI + Cloudflare Tunnel
Run each cell in order. The last cell outputs your public access URL.

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, VRAM: {p.total_memory / 1024**3:.1f} GB')

In [ ]:
%%bash
set -e
pip install -q huggingface_hub hf_transfer
CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install -q llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
pip install -q open-webui
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared
echo 'All dependencies installed.'

In [ ]:
import os
from huggingface_hub import hf_hub_download

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

MODEL_DIR = '/kaggle/working/models'
MODEL_FILE = 'qwen2.5-32b-instruct-q4_k_m.gguf'
MODEL_PATH = os.path.join(MODEL_DIR, MODEL_FILE)
os.makedirs(MODEL_DIR, exist_ok=True)

if os.path.exists(MODEL_PATH):
    size_gb = os.path.getsize(MODEL_PATH) / 1024**3
    print(f'Model already exists: {size_gb:.1f} GB, skipping download.')
else:
    print('Downloading model (~20GB), please wait 15-25 minutes...')
    hf_hub_download(
        repo_id='Qwen/Qwen2.5-32B-Instruct-GGUF',
        filename=MODEL_FILE,
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False
    )
    print(f'Download complete: {MODEL_PATH}')

In [ ]:
import subprocess, time, requests

MODEL_PATH = '/kaggle/working/models/qwen2.5-32b-instruct-q4_k_m.gguf'
LLAMA_PORT = 8080

cmd = [
    'python', '-m', 'llama_cpp.server',
    '--model', MODEL_PATH,
    '--host', '0.0.0.0',
    '--port', str(LLAMA_PORT),
    '--n_gpu_layers', '99',
    '--n_ctx', '32768',
    '--n_batch', '512',
    '--tensor_split', '1,1',
    '--chat_format', 'chatml',
    '--verbose', 'false'
]

print('Starting llama.cpp server...')
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print('Loading model, please wait up to 60 seconds...')
for i in range(45):
    time.sleep(2)
    try:
        r = requests.get(f'http://localhost:{LLAMA_PORT}/v1/models', timeout=3)
        if r.status_code == 200:
            print(f'llama.cpp server is ready on port {LLAMA_PORT}')
            break
    except:
        print(f'  Loading... {(i+1)*2}s', end='\r')
else:
    print('Timeout. Check error output:')
    print(proc.stdout.read(3000))

In [ ]:
import subprocess, time, requests, os

WEBUI_PORT = 3000
env = os.environ.copy()
env.update({
    'OLLAMA_BASE_URL': '',
    'OPENAI_API_BASE_URL': 'http://localhost:8080/v1',
    'OPENAI_API_KEY': 'sk-no-key',
    'WEBUI_AUTH': 'False',
    'PORT': str(WEBUI_PORT),
    'HOST': '0.0.0.0'
})

print('Starting Open WebUI...')
webui_proc = subprocess.Popen(
    ['open-webui', 'serve'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for i in range(30):
    time.sleep(2)
    try:
        r = requests.get(f'http://localhost:{WEBUI_PORT}', timeout=3)
        if r.status_code == 200:
            print(f'Open WebUI is ready on port {WEBUI_PORT}')
            break
    except:
        print(f'  Starting... {(i+1)*2}s', end='\r')
else:
    print('Timeout. Check error output:')
    print(webui_proc.stdout.read(3000))

In [ ]:
import subprocess, threading, re

cf_url = None
url_found = threading.Event()

def read_output(proc):
    global cf_url
    for line in proc.stderr:
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if m:
            cf_url = m.group(0)
            url_found.set()

print('Starting Cloudflare Tunnel...')
cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:3000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

t = threading.Thread(target=read_output, args=(cf_proc,), daemon=True)
t.start()

if url_found.wait(timeout=30):
    print('=' * 50)
    print('Tunnel ready! Open this URL on your phone:')
    print(f'  {cf_url}')
    print('=' * 50)
    print('Keep this notebook running. URL changes on restart.')
else:
    print('Timeout getting URL. Error output:')
    print(cf_proc.stderr.read(2000))

## Notes

**If VRAM error:** Change `n_ctx` to `16384` in Step 4.

**Save model to avoid re-downloading:**
After download completes, go to the right panel -> Add Data -> New Dataset,
save `/kaggle/working/models/` as a private dataset.
Next time, mount that dataset and change MODEL_PATH to `/kaggle/input/your-dataset-name/qwen2.5-32b-instruct-q4_k_m.gguf`.

**Cloudflare warning page:** Click 'Click to proceed' to enter the chat interface.

**System Prompt for novel writing (paste in Open WebUI Settings):**
```
You are a professional Chinese novel writer. Keep characters consistent.
Write vivid scenes with sensory details. Each chapter should end with a hook.
Generate 1500-2000 Chinese characters per response. Start writing immediately.
```